# Code execution + Files API

Two related server-side primitives:

- **Files API**: upload a file ahead of time to Anthropic.
  You get back a `file_id` you can reference from any later request.
- **Code execution tool**: a sandboxed Python container runs code Claude writes,
  and streams the results back as a `code_execution_tool_result` block.

The two combine naturally: upload a dataset, then ask Claude to analyze it.
Claude writes Python, runs it in the container (no network, isolated),
and returns the analysis plus any generated plots as new files.

## Beta headers

Both features are behind beta flags.
Pass them to `chat` via `betas=["code-execution-2025-08-25", "files-api-2025-04-14"]`.
`chat_tools.chat` routes to the beta endpoint when `betas` is set.

## Response shape

`response.content` is a list of mixed blocks:
- `text` — Claude's narration.
- `server_tool_use` — the code Claude wrote.
- `bash_code_execution_tool_result` / `text_editor_code_execution_tool_result` — sandbox stdout, stderr,
  and `file_id`s for any files the sandbox produced.

## File lifecycle

Files uploaded via `upload_file` persist on Anthropic's side;
download with `download_file(id)` and clean up with `delete_file(id)` when you no longer need them.

Model: `claude-sonnet-4-6`.


In [1]:
from chat_tools import add_user_message, chat
from files_api import delete_file, download_file, upload_file
from models import MODEL_SONNET

In [2]:
# Upload the CSV once; the file_id is reusable across requests until deleted.
file_metadata = upload_file("streaming.csv")
print(f"uploaded: id={file_metadata.id}, name={file_metadata.filename}")

uploaded: id=file_01QAzT1CXQ2pqP9W5piHMvgU, name=streaming.csv


In [3]:
analysis_prompt = (
    "Run a detailed analysis to determine major drivers of churn. "
    "Your final output should include at least one detailed plot summarizing your findings. "
    "Critical note: every time you execute code, you're starting with a completely clean slate. "
    "No variables or library imports from previous executions exist. Redeclare / reimport as needed."
)

text_block = {"type": "text", "text": analysis_prompt}
upload_block = {"type": "container_upload", "file_id": file_metadata.id}

messages = []
add_user_message(messages, [text_block, upload_block])

response = chat(
    messages,
    model=MODEL_SONNET,
    max_tokens=10000,
    tools=[{"type": "code_execution_20250825", "name": "code_execution"}],
    betas=["code-execution-2025-08-25", "files-api-2025-04-14"],
)

for block in response.content:
    print(f"--- {block.type} ---")
    if block.type == "text":
        print(block.text)
    elif block.type == "server_tool_use":
        print(f"tool: {block.name}")
        print(str(block.input)[:400])
    elif block.type == "code_execution_tool_result":
        print(str(block.content)[:400])
    else:
        print(str(block)[:300])
    print()

--- server_tool_use ---
tool: bash_code_execution
{'command': 'ls $INPUT_DIR && head -50 $INPUT_DIR/streaming.csv'}

--- bash_code_execution_tool_result ---
BetaBashCodeExecutionToolResultBlock(content=BetaBashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='streaming.csv\nUserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,Cus

--- server_tool_use ---
tool: bash_code_execution
{'command': 'wc -l $INPUT_DIR/streaming.csv && python3 -c "\nimport pandas as pd\ndf = pd.read_csv(\'$INPUT_DIR/streaming.csv\')\nprint(\'Shape:\', df.shape)\nprint(\'\\nDtypes:\\n\', df.dtypes)\nprint(\'\\nMissing values:\\n\', df.isnull().sum())\nprint(\'\\nBasic stats:\\n\', df.describe())\nprint(\'\\nChurn rate:\', df[\'Churned\'].mean())\nprint(\'\\nSubscription tier dist:\\n\', df[\'Subscrip

--- bash_code_execution_tool_result ---
BetaBashCodeExecutionToolResultBlock(content=Beta

In [4]:
# Walk the response for file_ids produced by the sandbox and download each one.
# The beta code_execution_20250825 returns block types like `bash_code_execution_tool_result`
# and `text_editor_code_execution_tool_result`, so match by suffix.
generated_ids = []
for block in response.content:
    if not block.type.endswith("code_execution_tool_result"):
        continue
    for item in getattr(block.content, "content", None) or []:
        file_id = getattr(item, "file_id", None)
        if file_id:
            filename = download_file(file_id)
            generated_ids.append(file_id)
            print(f"downloaded {file_id} -> {filename}")

downloaded file_01GkvkcU2gh4LArb1aWhzczd -> churn_analysis.png


In [ ]:
# Clean up the CSV we uploaded and any files the sandbox generated.
delete_file(file_metadata.id)
for file_id in generated_ids:
    delete_file(file_id)